In [ ]:
!pip install vllm

In [ ]:
import base64
import os
from typing import List, Optional, Tuple, Union
from vllm import LLM, SamplingParams

INFO 11-18 19:06:13 [__init__.py:216] Automatically detected platform cpu.


ImportError: cannot import name 'ImagePixelInputs' from 'vllm.multimodal.image' (/Users/artemdzalilov/working/VSEROSII_first/venv/lib/python3.11/site-packages/vllm/multimodal/image.py)

In [ ]:
class VLLMModel:
    def __init__(
        self, 
        model_name: str, 
        gpu_memory_utilization: float = 0.90,
        max_model_len: int = 4096,
        temperature: float = 0.0,
        max_tokens: int = 1024
    ):
        """
        Инициализация модели vLLM.
        """
        self.model_name = model_name
        
        # Инициализируем движок vLLM
        self.llm = LLM(
            model=model_name,
            max_model_len=max_model_len,
            gpu_memory_utilization=gpu_memory_utilization,
            trust_remote_code=True
        )

        # Базовые параметры генерации
        self.default_sampling_params = SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens
        )

    def _encode_image_base64(self, image_path: str) -> str:
        """Читает файл и переводит в base64 строку."""
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")
            
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def _prepare_messages(self, prompt: str, image_path: Optional[str] = None) -> List[dict]:
        """Формирует структуру сообщений для chat-интерфейса vLLM."""
        content = []
        
        # Добавляем текст
        content.append({"type": "text", "text": prompt})
        
        # Добавляем изображение, если путь передан
        if image_path:
            image_b64 = self._encode_image_base64(image_path)
            # Определяем расширение для MIME-типа (упрощенно)
            ext = image_path.split('.')[-1].lower()
            mime = "png" if ext == "png" else "jpeg"
            
            content.append({
                "type": "image_url", 
                "image_url": {"url": f"data:image/{mime};base64,{image_b64}"}
            })

        return [
            {"role": "user", "content": content}
        ]

    def invoke(self, text: str, image_path: Optional[str] = None) -> str:
        """
        Запуск инференса для одного примера.
        
        :param text: Текстовый запрос
        :param image_path: Путь до файла изображения (опционально)
        :return: Сгенерированный текст ответа
        """
        messages = self._prepare_messages(text, image_path)
        
        # vLLM принимает список бесед, поэтому оборачиваем messages в список [messages]
        outputs = self.llm.chat(
            messages=messages, 
            sampling_params=self.default_sampling_params
        )
        
        return outputs[0].outputs[0].text.strip()

    def invoke_batch(
        self, 
        inputs: List[Tuple[str, Optional[str]]], 
        batch_size: int = 8
    ) -> List[str]:
        """
        Запуск инференса батчами.
        
        :param inputs: Список кортежей (текст, путь_до_картинки). 
                       Если картинки нет, второй элемент кортежа должен быть None.
        :param batch_size: Размер батча для подачи в модель.
        :return: Список ответов (строк) в том же порядке.
        """
        results = []
        
        # Проходим по списку входных данных с шагом batch_size
        for i in range(0, len(inputs), batch_size):
            batch_chunk = inputs[i : i + batch_size]
            
            # Подготовка списка сообщений для текущего батча
            # vLLM поддерживает подачу списка conversations для параллельной обработки
            batch_messages = []
            for text, img_path in batch_chunk:
                batch_messages.append(self._prepare_messages(text, img_path))
            
            # Запуск генерации для батча
            outputs = self.llm.chat(
                messages=batch_messages,
                sampling_params=self.default_sampling_params
            )
            
            # Извлечение текстов из результатов
            batch_results = [output.outputs[0].text.strip() for output in outputs]
            results.extend(batch_results)
            
        return results


INFO 11-18 19:50:50 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 11-18 19:50:51 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 11-18 19:50:51 [model.py:1510] Using max model len 4096
INFO 11-18 19:50:51 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.
INFO 11-18 19:50:56 [__init__.py:216] Automatically detected platform cpu.
(EngineCore_DP0 pid=97283) INFO 11-18 19:50:56 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=97283) INFO 11-18 19:50:56 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.38s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.38s/it]
(EngineCore_DP0 pid=97283) 


(EngineCore_DP0 pid=97283) INFO 11-18 19:51:00 [default_loader.py:267] Loading weights took 1.38 seconds
(EngineCore_DP0 pid=97283) INFO 11-18 19:51:00 [kv_cache_utils.py:1087] GPU KV cache size: 37,440 tokens
(EngineCore_DP0 pid=97283) INFO 11-18 19:51:00 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 9.14x
(EngineCore_DP0 pid=97283) INFO 11-18 19:51:01 [cpu_model_runner.py:117] Warming up model for the compilation...
(EngineCore_DP0 pid=97283) WARNING 11-18 19:51:01 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=97283) INFO 11-18 19:51:16 [cpu_model_runner.py:121] Warming up done.
(EngineCore_DP0 pid=97283) INFO 11-18 19:51:16 [core.py:210] init engine (profile, create kv cache, warmup model) took 15.71 seconds
(EngineCore_DP0 pid=97283) WARNING 11-18 19:51:17 [cpu.py:117] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 11-18

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Answer: <think>
Хорошо, пользователь спрашивает, сколько будет 2+2. Надо подумать, что он хочет узнать. Возможно, он просто выполняет операцию, но нужно уточнить, что именно он хочет. Может быть, он хочет проверить, что 2+2 действительно равно 4, или он имеет другую цель. В любом случае, ответ должен быть логичным и соответствующим.

Пользователь может быть просто вежливым, и он не хочет усложнять ситуацию. Возможно, он хочет убедиться, что его ответ правильный. В таком случае, ответ 4. Но нужно убедиться, что нет других возможных интерпретаций. Например, если он имеет другие числа, но в данном случае только 2+2. Значит, правильный ответ 4.
</think>

2 + 2 = 4.

--- Batch Processing ---


Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Q: What is capital of France?
A: <think>
Okay, the user is asking about the capital of France. Let me start by recalling what I know. France's capital is Paris. I think that's correct. But wait, maybe I should double-check. I remember that Paris is the main city in France, and it's also the capital. No, wait, no, wait. Wait, no, wait. Wait, no, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. Wait, no, no. W

In [ ]:
if __name__ == "__main__":
    # Путь к вашей модели
    MODEL_PATH = "Qwen/Qwen3-0.6B" 
    
    # Инициализация
    model = VLLMModel(model_name=MODEL_PATH)

    # 1. Одиночный запуск (только текст)
    print("--- Single Text ---")
    res1 = model.invoke("Сколько будет 2+2?")
    print("Answer:", res1)

    # 2. Одиночный запуск (текст + картинка)
    # print("--- Single Image ---")
    # res2 = model.invoke("Describe this image", image_path="test.jpg")
    # print("Answer:", res2)

    # 3. Пакетная обработка
    print("\n--- Batch Processing ---")
    batch_data = [
        ("What is capital of France?", None),
        ("Explain quantum physics in one sentence", None),
        # ("What is in this picture?", "cat.jpg"), 
    ]
    
    batch_results = model.invoke_batch(batch_data, batch_size=2)
    for q, a in zip(batch_data, batch_results):
        print(f"Q: {q[0]}\nA: {a}\n")